Install Dependencies

In [1]:
!pip install transformers flask flask-cors pyngrok peft

Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Log in to Hugging Face

In [3]:
from huggingface_hub import login
login()


Load Model and Tokenizer

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

# Path to your LoRA adapter in Google Drive
peft_path = "/content/drive/MyDrive/mistral-whatsapp-final"

# Load base Mistral model from Hugging Face (requires login)
base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.1",
    device_map="auto",
    torch_dtype=torch.float16
)

# Load tokenizer from the base model
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")

# Load the fine-tuned LoRA adapter
model = PeftModel.from_pretrained(base_model, peft_path)

# Move to GPU or CPU
model.to("cuda" if torch.cuda.is_available() else "cpu")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Line

Set up Flask Server

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

@app.route("/generate", methods=["POST"])
def generate():
    data = request.json
    prompt = data.get("prompt", "")

    # Add Gurt's name to steer the model into speaking as the bot
    prompt += "\nGurt:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.85,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract just Gurt's reply (first line only)
    if "Gurt:" in full_output:
        reply = full_output.split("Gurt:")[-1].strip().split("\n")[0]
    else:
        reply = full_output[len(prompt):].strip().split("\n")[0]

    return jsonify({"response": reply})


Authenticate Ngrok Authtoken

In [6]:
!ngrok config add-authtoken 2wvJt5WodUEHn0II3UzJL09WmwD_6cCoyQvPMqueu4uyuJyQq

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


Run with Ngrok

In [7]:
from pyngrok import ngrok

public_url = ngrok.connect(5000)
print("✅ Public URL:", public_url)

app.run(port=5000)


✅ Public URL: NgrokTunnel: "https://7d13-34-124-184-66.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [10/May/2025 22:42:17] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/May/2025 22:48:55] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/May/2025 22:51:53] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/May/2025 22:52:00] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/May/2025 22:52:06] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/May/2025 22:52:11] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/May/2025 22:52:18] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/May/2025 22:52:20] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/May/2025 22:52:29] "POST /generate HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/May/2025 22:52:32